In [ ]:
!pip install -q sentence-transformers pandas numpy tqdm rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 21.8 MB/s eta 0:00:00


In [ ]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch

with open('../../rzd_locations.json', 'r', encoding='utf-8') as file:
  rzd_data = json.load(file)

with open('../data/yandex_rasp_locations.json', 'r', encoding='utf-8') as file:
  yandex_data = json.load(file)

# TODO: следующий шаг для улучшения - получить full_address для станций
# у алгоритма большие проблемы с такого рода случаями из-за совершенно разной записи адреса
# rzd_data = [
#     {
#     "nodeId": "5bee9c834aa8314521c18617",
#     "cityId": "5a323c29340c7441a0a556bb",
#     "expressCode": "2000998",
#     "name": "Бекасово 1",
#     "nodeType": "station",
#     "transportType": "train",
#     "region": "Москва, Российская Федерация",
#     "regionIso": "RU-MOW",
#     "countryIso": "RU",
#     "suburbanCode": "3699",
#     "foreignCode": "2000998"
#   }
# ]

# yandex_data = [
#   {
#     "name": "Бекасово-1",
#     "city": "Бекасово",
#     "region": "Москва и Московская область",
#     "country": "Россия",
#     "station_type": "station",
#     "transport_type": "train",
#     "yandex_code": "s9601625",
#     "esr_code": "180010",
#     "lat": 55.429644,
#     "lon": 36.840156,
#     "source": "yandex_rasp_api"
#   }
# ]

In [ ]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch

rzd_data = [
   {
    "nodeId": "5a8ac9cb340c742584d3678a",
    "cityId": "5a13d0c9340c745ca1ed1e61",
    "expressCode": "2078775",
    "name": "Семь Колодезей",
    "nodeType": "station",
    "transportType": "train",
    "region": "Ленино, Ленинский Район, Крым Республика, Российская Федерация",
    "regionIso": "RU-CR",
    "countryIso": "RU",
    "suburbanCode": "42304",
    "foreignCode": "2210775"
  }
]

yandex_data = [
  {
    "name": "Семь Колодезей",
    "city": "Ленино",
    "region": "Крым",
    "country": "Россия",
    "station_type": "station",
    "transport_type": "train",
    "yandex_code": "s9617027",
    "esr_code": "471508",
    "lat": 45.299209,
    "lon": 35.772853,
    "source": "yandex_rasp_api"
  }
]

In [ ]:
# ========== ПРЕДОБРАБОТКА ==========

def clean_name(name):
    """Очистка и нормализация названия"""
    import re
    name = name.lower().strip()
    name = name.replace('ё', 'е')
    name = re.sub(r'\s+', ' ', name)
    if re.search(r'\d+ км', name):
        name = re.sub(r'\b(ост\. пункт|ост\.пункт|ост пункт|ост\.пунтк|оп|разъезд|рзд|платформа|путевой пост|тупик|мост|переезд|блок пост|пост|блокпост|обгонный пункт|казарма|граница)\b', '', name)
    return name.strip()


def clean_address(address):
    import re
    address = address.lower()
    address = re.sub(r'\s+', ' ', address)
    # address = re.sub(r'Городской округ \w+, ', '', address)
    return address


def clean_settlement_name(name):
    """Нормализует название населённого пункта для точного поиска по имени.

    Убирает служебные слова (г., город, село, деревня, посёлок, пгт и т.п.),
    удаляет уточняющие фразы (вокзал, автостанция и т.п.),
    обрезает всё после запятой/тире и оставляет только буквы и пробелы.
    """
    import re
    if not name:
        return ''
    s = clean_name(name)

    # Удаляем начальные префиксы "г.", "город"
    s = re.sub(r'^(г\.|г\b|город\b)\s*', '', s)

    # Удаляем служебные слова внутри строки
    s = re.sub(r'\b(село|деревня|поселок|посёлок|им|имени|пгт|ст-ца|рп|пгт|с\.|д\.|п)\b', '', s)

    # Удаляем слова, указывающие на тип здания/остановки
    s = re.sub(r'\b(автостанция|авт\.ст\.|авт\.ост\.|вокзал|вокз\.|вокз|станция|станции|остров|ост)\b', '', s)

    # дефис на пробел
    s = re.sub(r'-', ' ', s)

    # Удаляем все уточнения в скобках
    s = re.sub(r'\(.+\)', '', s)

    # Оставляем только буквы (русские/латиница) и цифры и пробелы
    s = re.sub(r'[^a-z0-9а-яё\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def normalize_rzd_transport_type(item):
    transport_type = (item.get('transportType') or '').strip().lower()
    node_type = (item.get('nodeType') or '').strip().lower()

    transport_aliases = {
        'avia': 'plane',
        'suburban': 'train',
        'boat': 'water',
    }

    if transport_type in transport_aliases:
        return transport_aliases[transport_type]

    if transport_type in {'city', 'plane', 'train', 'bus', 'water'}:
        return transport_type
    if node_type == 'city':
        return 'city'
    if node_type == 'station':
        return 'train'
    return ''

def normalize_yandex_transport_type(item):
    transport_type = (item.get('transport_type') or '').strip().lower()
    station_type = (item.get('station_type') or '').strip().lower()

    transport_aliases = {
        'suburban': 'train',
        'helicopter': 'plane',
    }

    if transport_type in transport_aliases:
        return transport_aliases[transport_type]

    if transport_type in {'city', 'plane', 'train', 'bus', 'water'}:
        return transport_type

    station_to_transport = {
        'city': 'city',

        # railway
        'station': 'train',
        'platform': 'train',
        'stop': 'train',
        'checkpoint': 'train',
        'post': 'train',
        'crossing': 'train',
        'overtaking_point': 'train',
        'train_station': 'train',
        'suburban_station': 'train',

        # air
        'airport': 'plane',

        # bus
        'bus_station': 'bus',
        'bus_stop': 'bus',

        # water
        'port': 'water',
        'port_point': 'water',
        'wharf': 'water',
        'river_port': 'water',
        'marine_station': 'water',
    }

    if station_type == 'unknown':
        return ''

    return station_to_transport.get(station_type, '')

# ========== ДОПОЛНИТЕЛЬНЫЕ УТИЛИТЫ ДЛЯ ВЕСОВОГО СКОРА ==========
import re

def parse_address_parts(name, region, is_city_address, city_name=None):
    """Разбирает адрес на city/district/region (упрощённо, лексически)."""
    if not region:
        return {}

    value = clean_name(region)
    value = re.sub(r"[;/]", ",", value)
    value = re.sub(r"\b(российская федерация|россия|рф)\b", "", value)

    parts_raw = [part.strip() for part in value.split(",") if part.strip()]
    city_key = clean_name(city_name) if city_name else ""

    region_marker = re.compile(r"\b(область|обл|край|республика|респ|ао|автономный округ|округ)\b")
    district_marker = re.compile(r"\b(район|р-н|городской округ)\b")
    city_marker = re.compile(r"\b(город|г\.|г)\b")
    noise_words = re.compile(
        r"\b(область|обл|край|республика|респ|ао|автономный округ|округ|"
        r"район|р-н|муниципальный|городской|городского|муниципального|г\.|г|город|поселение|поселок|посёлок\b"
    )

    parts = {}

    cleaned_name = clean_name(name)
    cleaned_city_name = clean_name(city_name or "")

    # TODO: настроить ненужные добавки РЖД в строку региона
    if len(parts_raw) > 0 and "и(при) станция(и)" in parts_raw[0]:
      parts_raw.pop(0)

    if city_name and not (is_city_address and cleaned_city_name == cleaned_name):
        parts["city"] = cleaned_city_name

    for part in parts_raw:
        has_region = bool(region_marker.search(part))
        has_district = bool(district_marker.search(part))
        has_city = bool(city_marker.search(part))

        cleaned = noise_words.sub("", part)
        cleaned = re.sub(r"[^a-z0-9а-яё\s-]", " ", cleaned)
        cleaned = re.sub(r"\s+", " ", cleaned).strip()

        if not cleaned:
            continue

        if city_key and (cleaned == city_key or (city_key in cleaned and not has_region and not has_district)):
            parts.setdefault("city", cleaned)
            continue

        if has_city and "city" not in parts:
            parts["city"] = cleaned
            continue
        if has_district and "district" not in parts:
            parts["district"] = cleaned
            continue
        if has_region and "region" not in parts:
            parts["region"] = cleaned
            continue

        if "city" not in parts:
            parts["city"] = cleaned
        elif "district" not in parts:
            parts["district"] = cleaned
        elif "region" not in parts:
            parts["region"] = cleaned

    if "region" in parts:
        if parts["region"] == "москва и московская":
            parts["region"] = "московская"
        elif parts["region"] == "санкт-петербург и ленинградская":
            parts["region"] = "ленинградская"
        elif parts["region"] == "ханты-мансийский - югра":
            parts["region"] = "ханты-мансийский"
        elif parts["region"] == "саха якутия":
            parts["region"] = "саха-якутия"
        parts["region"] = parts["region"].replace(" - кузбасс", "")
        parts["region"] = parts["region"].replace(" - чувашия", "")

    if is_city_address:
      if parts.get("city") == clean_name(name):
        parts.pop("city", None)

    return parts

def extract_km(name):
    if not name:
        return None
    m = re.search(r"(\d+)\s*км", str(name).lower())
    return int(m.group(1)) if m else None

def string_similarity(a, b):
    from rapidfuzz import fuzz
    """Лёгкая строковая похожесть [0..1]."""
    if not a or not b:
        return None
    return fuzz.ratio(a, b) / 100.0


def weighted_similarity(name_score, rzd_parts, yandex_parts, weights):
    """Взвешенный скор: name cosine + строковые части адреса."""
    total_weight = weights["name"]
    total_score = weights["name"] * name_score

    for key in ("region", "district", "city"):
        first_key = rzd_parts.get(key)
        second_key = yandex_parts.get(key)

        # штраф за пустой аналог
        if (not first_key and second_key) or (first_key and not second_key):
          total_score -= 0.01
        else:
          score = string_similarity(first_key, second_key)
          if score is not None:
              w = weights.get(key, 0.0)
              total_weight += w
              total_score += w * score

    return total_score / total_weight if total_weight > 0 else name_score


def build_name_string(item):
    """Для эмбеддингов используем только имя (без региона)."""
    return clean_name(item.get("name", ""))

In [ ]:
# ========== ЗАГРУЗКА МОДЕЛИ ==========

print("🔄 Загрузка модели multilingual-e5-large...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"📱 Используется устройство: {device}")

model = SentenceTransformer('intfloat/multilingual-e5-large', device=device)

🔄 Загрузка модели multilingual-e5-large...
📱 Используется устройство: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
# ========== КОДИРОВАНИЕ ==========
print("📊 Подготовка строк для эмбеддинга...")

rzd_data_cities = [item for item in rzd_data if normalize_rzd_transport_type(item) == 'city']
yandex_data_cities = [item for item in yandex_data if normalize_yandex_transport_type(item) == 'city']

rzd_data_stations = [item for item in rzd_data if normalize_rzd_transport_type(item) != 'city']
yandex_data_stations = [item for item in yandex_data if normalize_yandex_transport_type(item) != 'city']

def get_similarity_matrix(data1, data2):
    if len(data1) == 0 or len(data2) == 0:
        return np.empty((len(data1), len(data2)))

    strings1 = [build_name_string(item, 'rzd') for item in data1]
    strings2 = [build_name_string(item, 'yandex') for item in data2]

    embeddings1 = model.encode(
        strings1,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    embeddings2 = model.encode(
        strings2,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    print("🔍 Расчёт матрицы сходства...")
    from sklearn.metrics.pairwise import cosine_similarity
    similarity_matrix = cosine_similarity(embeddings1, embeddings2)
    return similarity_matrix

similarity_cities = get_similarity_matrix(rzd_data_cities, yandex_data_cities)
similarity_stations = get_similarity_matrix(rzd_data_stations, yandex_data_stations)

📊 Подготовка строк для эмбеддинга...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Расчёт матрицы сходства...


In [ ]:
def build_address_parts_rzd(items):
    parts = []
    for item in items:
        is_city = normalize_rzd_transport_type(item) == "city"
        name = item.get("name", "")
        city_for_region = name if is_city else ""
        parts.append(parse_address_parts(name, item.get("region", ""), is_city, city_for_region))
    return parts

def build_address_parts_yandex(items):
    parts = []
    for item in items:
        city_for_region = item.get("city", "")
        is_city = normalize_yandex_transport_type(item) == "city"
        name = item.get("name", "")
        region_source = item.get("full_address") or f"{item.get('region', '')}"
        parts.append(parse_address_parts(name, region_source, is_city, city_for_region))
    return parts

In [ ]:
build_address_parts_rzd([{
    "nodeId": "5e05af357081a630d452d556",
    "cityId": "5a13bb25340c745ca1e80dc5",
    "expressCode": "2060206",
    "name": "Ост.Пункт 6 Км",
    "nodeType": "station",
    "transportType": "train",
    "region": "Чебоксары, Российская Федерация",
    "regionIso": "RU-CU",
    "countryIso": "RU",
    "suburbanCode": "41260"
  }])

[{'city': 'чебоксары'}]

In [ ]:
build_address_parts_yandex([{
    "name": "6 км",
    "city": "Чебоксары",
    "region": "Чувашская Республика",
    "country": "Россия",
    "station_type": "stop",
    "transport_type": "train",
    "yandex_code": "s9657395",
    "esr_code": "001388",
    "lat": 56.077138870201,
    "lon": 47.2242203278802,
    "source": "yandex_rasp_api"
  }])

[{'city': 'чебоксары', 'region': 'чувашская'}]

In [ ]:
rzd_parts_cities = build_address_parts_rzd(rzd_data_cities)
yandex_parts_cities = build_address_parts_yandex(yandex_data_cities)

rzd_parts_stations = build_address_parts_rzd(rzd_data_stations)
yandex_parts_stations = build_address_parts_yandex(yandex_data_stations)

In [ ]:
rzd_parts_stations

[{'city': 'ленино', 'district': 'ленинский', 'region': 'крым'}]

In [ ]:
yandex_parts_stations

[{'city': 'ленино', 'district': 'крым'}]

In [ ]:
WEIGHTS = {
    "name": 0.4,
    "region": 0.3,
    "district": 0.2,
    "city": 0.1,
}
weighted_similarity(1.0, rzd_parts_stations[0], yandex_parts_stations[0], WEIGHTS)

0.7884615384615384

In [ ]:
# ========== ПОИСК ЛУЧШИХ СООТВЕТСТВИЙ ==========
def find_matches(
    sim_matrix,
    rzd_data,
    yandex_data,
    rzd_parts,
    yandex_parts,
    threshold=0.85,
    weights=WEIGHTS,
):
    """Находит лучшие соответствия выше порога с учетом совместимости типов.
    Использует взвешенный скор: name embeddings + string similarity адресных частей.
    Учитывает уникальность yandex_code — если лучший уже занят, берём следующий по скору.
    """
    matches = []

    if sim_matrix.shape[1] == 0:
        return matches

    yandex_indices_by_type = {}
    for j, item in enumerate(yandex_data):
        y_type = normalize_yandex_transport_type(item)
        yandex_indices_by_type.setdefault(y_type, []).append(j)

    yandex_indices_by_name = {}
    for j, item in enumerate(yandex_data):
        base = clean_settlement_name(item.get('name', ''))
        if base:
            yandex_indices_by_name.setdefault(base, []).append(j)

    used_yandex_codes = set()

    def pick_best_candidate(candidate_indices, i):
        scored = []
        rzd_name = rzd_data[i].get("name", "")
        rzd_km = extract_km(rzd_name)

        for j in candidate_indices:
            y_code = yandex_data[j].get('yandex_code')
            if y_code in used_yandex_codes:
                continue

            # если в названии станции есть км — номера должны совпасть
            if normalize_yandex_transport_type(yandex_data[j]) != "city":
                y_name = yandex_data[j].get("name", "")
                y_km = extract_km(y_name)
                if rzd_km is not None and y_km is not None:
                    if rzd_km != y_km:
                        continue  # обнуляем similarity, пропускаем

            name_score = sim_matrix[i][j]
            score = weighted_similarity(name_score, rzd_parts[i], yandex_parts[j], weights)
            scored.append((score, j))

        if not scored:
            return None, None
        scored.sort(reverse=True, key=lambda x: x[0])
        return scored[0][1], scored[0][0]

    # --- СОРТИРОВКА RZD ПО ЛУЧШЕМУ ДОСТИЖИМОМУ СКОРУ ---
    rzd_ranked = []
    for i in range(len(rzd_data)):
        rzd_type = normalize_rzd_transport_type(rzd_data[i])
        compatible_indices = yandex_indices_by_type.get(rzd_type, [])
        if not compatible_indices:
            continue

        candidate_indices = compatible_indices
        if rzd_type == "city":
            base_name = clean_settlement_name(rzd_data[i].get('name', ''))
            if base_name and base_name in yandex_indices_by_name:
                candidate_indices = [
                    j for j in yandex_indices_by_name[base_name]
                    if j in compatible_indices
                ] or compatible_indices

        # лучший возможный score (без учета used_yandex_codes)
        best_score = None
        for j in candidate_indices:
            name_score = sim_matrix[i][j]
            score = weighted_similarity(name_score, rzd_parts[i], yandex_parts[j], weights)
            if best_score is None or score > best_score:
                best_score = score

        if best_score is not None:
            rzd_ranked.append((best_score, i))

    rzd_ranked.sort(reverse=True, key=lambda x: x[0])
    rzd_order = [i for _, i in rzd_ranked]
    # ---------------------------------------------------

    for i in rzd_order:
        rzd_type = normalize_rzd_transport_type(rzd_data[i])
        compatible_indices = yandex_indices_by_type.get(rzd_type, [])
        if not compatible_indices:
            continue

        # Для городов — сначала проверяем совпадение нормализованного имени
        if rzd_type == 'city':
            base_name = clean_settlement_name(rzd_data[i].get('name', ''))
            same_name_candidates = []
            if base_name:
                same_name_candidates = [j for j in yandex_indices_by_name.get(base_name, []) if j in compatible_indices]

            if same_name_candidates:
                best_j, best_score = pick_best_candidate(same_name_candidates, i)
                if best_j is not None and best_score >= threshold:
                    used_yandex_codes.add(yandex_data[best_j].get('yandex_code'))
                    matches.append({
                        'rzd_name': rzd_data[i]['name'],
                        'rzd_region': rzd_data[i].get('region', ''),
                        'rzd_type': rzd_type,
                        'expressCode': rzd_data[i].get('expressCode'),
                        'yandex_name': yandex_data[best_j]['name'],
                        'yandex_region': yandex_data[best_j].get('full_address', yandex_data[best_j].get('region', '')),
                        'yandex_type': normalize_yandex_transport_type(yandex_data[best_j]),
                        'similarity': round(float(best_score), 4),
                        'yandex_code': yandex_data[best_j].get('yandex_code')
                    })
            continue

        # Обычная логика — среди всех совместимых по типу
        best_j, best_score = pick_best_candidate(compatible_indices, i)
        if best_j is None:
            continue

        yandex_full_address = yandex_data[best_j].get('full_address')
        if not yandex_full_address:
            yandex_full_address = yandex_data[best_j].get('city', '') + ', ' + yandex_data[best_j].get('region', '')

        if best_score >= threshold:
            used_yandex_codes.add(yandex_data[best_j].get('yandex_code'))
            matches.append({
                'rzd_name': rzd_data[i]['name'],
                'rzd_region': rzd_data[i].get('region', ''),
                'rzd_type': rzd_type,
                'expressCode': rzd_data[i].get('expressCode'),
                'yandex_name': yandex_data[best_j]['name'],
                'yandex_region': yandex_full_address,
                'yandex_type': normalize_yandex_transport_type(yandex_data[best_j]),
                'similarity': round(float(best_score), 4),
                'yandex_code': yandex_data[best_j].get('yandex_code')
            })

    # return sorted(matches, key=lambda x: x['similarity'], reverse=True)
    return matches

# Ищем с разными порогами для демонстрации
# print("\n" + "="*80)
# print("📋 РЕЗУЛЬТАТЫ СОПОСТАВЛЕНИЯ")
# print("="*80)

# for threshold in [0.95]:
#     matches = find_matches(
#         similarity_cities,
#         rzd_data_cities,
#         yandex_data_cities,
#         rzd_parts_cities,
#         yandex_parts_cities,
#         threshold
#     )
#     if matches:
#         print(f"\n🔸 Порог сходства: {threshold} (найдено {len(matches)} совпадений)")
#         for m in matches[:10]:  # показываем первые 10",
#             print(f"\n  RZD: {m['rzd_name']} | {m['rzd_region']} ({m['rzd_type']})")
#             print(f"  YAN: {m['yandex_name']} | {m['yandex_region']} ({m['yandex_type']})")
#             print(f"  ✅ Сходство: {m['similarity']:.4f}")

# for threshold in [0.95]:
#     matches = find_matches(
#         similarity_stations,
#         rzd_data_stations,
#         yandex_data_stations,
#         rzd_parts_stations,
#         yandex_parts_stations,
#         threshold
#     )
#     if matches:
#         print(f"\n🔸 Порог сходства: {threshold} (найдено {len(matches)} совпадений)")
#         for m in matches[:10]:  # показываем первые 10",
#             print(f"\n  RZD: {m['rzd_name']} | {m['rzd_region']} ({m['rzd_type']})")
#             print(f"  YAN: {m['yandex_name']} | {m['yandex_region']} ({m['yandex_type']})")
#             print(f"  ✅ Сходство: {m['similarity']:.4f}")


In [ ]:
# ========== ЭКСПОРТ РЕЗУЛЬТАТОВ ==========

final_matches_cities = find_matches(
    similarity_cities,
    rzd_data_cities,
    yandex_data_cities,
    rzd_parts_cities,
    yandex_parts_cities,
    threshold=0.95
)

final_matches_stations = find_matches(
    similarity_stations,
    rzd_data_stations,
    yandex_data_stations,
    rzd_parts_stations,
    yandex_parts_stations,
    threshold=0.95
)

final_matches = final_matches_cities + final_matches_stations

# ========== ФУНКЦИЯ ДЛЯ ПОИСКА ЛУЧШЕГО ПОХОЖЕГО (ниже порога) ==========

def find_best_unmatched(similarity_matrix, rzd_data, yandex_data, matched_rzd_codes, matched_yandex_codes):
    """
    Для каждого ненайденного объекта находит самый похожий объект из другого источника.
    Возвращает список с информацией о ближайшем соседе.
    """
    unmatched_with_best = []

    # Ненайденные РЖД объекты
    for i, item in enumerate(rzd_data):
        if item.get('expressCode') not in matched_rzd_codes:
            best_j = np.argmax(similarity_matrix[i])
            best_score = similarity_matrix[i][best_j]

            yandex_full_address = yandex_data[best_j].get('full_address')

            if not yandex_full_address:
              yandex_full_address = yandex_data[best_j].get('region')

            unmatched_with_best.append({
                'source': 'rzd',
                'expressCode': item.get('expressCode'),
                'name': item.get('name'),
                'region': item.get('region'),
                'type': item.get('transportType') or item.get('nodeType'),
                'best_match_source': 'yandex',
                'best_match_name': yandex_data[best_j].get('name'),
                'best_match_region': yandex_full_address,
                'best_match_type': yandex_data[best_j].get('station_type'),
                'best_match_code': yandex_data[best_j].get('yandex_code'),
                'similarity': round(float(best_score), 4),
            })

    # Сортируем по убыванию похожести
    return sorted(unmatched_with_best, key=lambda x: x['similarity'], reverse=True)

matched_rzd_cities_codes = {m['expressCode'] for m in final_matches_cities}
matched_rzd_stations_codes = {m['expressCode'] for m in final_matches_stations}
matched_yandex_cities_codes = {m['yandex_code'] for m in final_matches_cities}
matched_yandex_stations_codes = {m['yandex_code'] for m in final_matches_stations}

not_matched_cities = find_best_unmatched(
    similarity_cities, rzd_data_cities, yandex_data_cities,
    matched_rzd_cities_codes, matched_yandex_cities_codes
)

not_matched_stations = find_best_unmatched(
    similarity_stations, rzd_data_stations, yandex_data_stations,
    matched_rzd_stations_codes, matched_yandex_stations_codes
)

all_not_matched = not_matched_cities + not_matched_stations

rzd_not_matched = [item for item in all_not_matched if item['source'] == 'rzd']
yandex_not_matched = [item for item in all_not_matched if item['source'] == 'yandex']


with open('matches.json', 'w', encoding='utf-8') as file:
    json.dump(final_matches, file, ensure_ascii=False, indent=2)

with open('not_matches.json', 'w', encoding='utf-8') as file:
    json.dump({
        'summary': {
            'total_rzd': len(rzd_data_cities) + len(rzd_data_stations) if 'rzd_data_cities' in locals() else len(rzd_data),
            'total_yandex': len(yandex_data_cities) + len(yandex_data_stations) if 'yandex_data_cities' in locals() else len(yandex_data),
            'matched_count': len(final_matches),
            'not_matched_total': len(all_not_matched),
            'rzd_not_matched': len(rzd_not_matched),
            'yandex_not_matched': len(yandex_not_matched)
        },
        'not_matched_items': all_not_matched,
    }, file, ensure_ascii=False, indent=2)

print("\n" + "="*80)
print("📊 ИТОГОВАЯ СТАТИСТИКА")
print("="*80)

total_rzd = len(rzd_data_cities) + len(rzd_data_stations) if 'rzd_data_cities' in locals() else len(rzd_data)
total_yandex = len(yandex_data_cities) + len(yandex_data_stations) if 'yandex_data_cities' in locals() else len(yandex_data)

print(f"✅ Matches: {len(final_matches)}")
print()
print(f"❌ Not matched всего: {len(all_not_matched)}")
print(f"   РЖД не нашло: {len(rzd_not_matched)}")
print(f"   Яндекс не нашло: {len(yandex_not_matched)}")

if rzd_not_matched:
    print("\n⚠️ Ненайденные, но с высоким сходством (первые 5):")
    print("-" * 70)
    for item in rzd_not_matched[:5]:
        print(f"  {item['source'].upper()}: {item['name']} ({item['region']})")
        print(f"  → Похож на: {item['best_match_name']} ({item['best_match_region']})")
        print(f"  Сходство: {item['similarity']:.4f}")
        print()


📊 ИТОГОВАЯ СТАТИСТИКА
✅ Matches: 7102

❌ Not matched всего: 4681
   РЖД не нашло: 4681
   Яндекс не нашло: 0

⚠️ Ненайденные, но с высоким сходством (первые 5):
----------------------------------------------------------------------
  RZD: Суна (Кондопожский Район, Карелия Республика, Российская Федерация)
  → Похож на: Суна (г. Суна, Кировская область)
  Сходство: 1.0000

  RZD: Лесной (Амурский Район, Хабаровский Край, Российская Федерация)
  → Похож на: Лесной (Лесной, Коломенский, Москва и Московская область)
  Сходство: 1.0000

  RZD: Лесной (Фировский Район, Тверская Область, Российская Федерация)
  → Похож на: Лесной (Лесной, Коломенский, Москва и Московская область)
  Сходство: 1.0000

  RZD: Лесной (Алтайский Край, Российская Федерация)
  → Похож на: Лесной (Лесной, Коломенский, Москва и Московская область)
  Сходство: 1.0000

  RZD: Лесной (Кувандыкский Район, Оренбургская Область, Российская Федерация)
  → Похож на: Лесной (Лесной, Коломенский, Москва и Московская область)
 